In [1]:

%pip install imbalanced-learn


In [2]:

import pandas as pd
import numpy as np
import joblib

from pathlib import Path
from scipy import sparse
from imblearn.combine import SMOTEENN

print("Imports successful!")


Imports successful!


In [13]:

from pathlib import Path
import pandas as pd
import numpy as np
from scipy import sparse

DATA_DIR = Path(
    "/content/drive/MyDrive/Colab Notebooks/data/processed"
)

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/Colab Notebooks/data/resampled"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def load_matrix(name):
    npy_path = DATA_DIR / f"{name}.npy"
    npz_path = DATA_DIR / f"{name}.npz"

    if npy_path.exists():
        return np.load(npy_path)

    if npz_path.exists():
        return sparse.load_npz(npz_path)

    raise FileNotFoundError(
        f"File not found: {npy_path} or {npz_path}"
    )

print("Processed directory exists:", DATA_DIR.exists())
print("Radiom training file exists:",
      (DATA_DIR / "Radiom_X_train.npy").exists())


Processed directory exists: True
Radiom training file exists: True


In [14]:

feature_groups = ["Radiom", "RadioClin", "RadioClinLab"]
targets = ["ICU", "mechanical_ventilation", "death"]

X_train_groups = {
    group: load_matrix(f"{group}_X_train")
    for group in feature_groups
}

X_test_groups = {
    group: load_matrix(f"{group}_X_test")
    for group in feature_groups
}

Y_train = pd.read_csv(DATA_DIR / "Y_train.csv")
Y_test = pd.read_csv(DATA_DIR / "Y_test.csv")

for group in feature_groups:
    print(
        f"{group}: train={X_train_groups[group].shape}, "
        f"test={X_test_groups[group].shape}"
    )

print("Labels loaded successfully.")


Radiom: train=(1680, 120), test=(720, 120)
RadioClin: train=(1680, 147), test=(720, 147)
RadioClinLab: train=(1680, 164), test=(720, 164)
Labels loaded successfully.


In [15]:

from imblearn.combine import SMOTEENN
import numpy as np
import pandas as pd

resampled_data = {}
resampling_summary = []

for group in feature_groups:
    X_train = X_train_groups[group]

    for target in targets:
        y_train = Y_train[target].astype(int)

        print(f"\nProcessing {group} | {target}")
        print("Before:", y_train.value_counts().to_dict())

        sampler = SMOTEENN(random_state=42)

        X_resampled, y_resampled = sampler.fit_resample(
            X_train,
            y_train
        )

        resampled_data[(group, target)] = (
            X_resampled,
            y_resampled
        )

        print(
            "After:",
            pd.Series(y_resampled).value_counts().to_dict()
        )

        resampling_summary.append({
            "Feature Group": group,
            "Target": target,
            "Original Rows": len(y_train),
            "Resampled Rows": len(y_resampled),
            "Original Positives": int(y_train.sum()),
            "Resampled Positives": int(np.sum(y_resampled))
        })

summary_df = pd.DataFrame(resampling_summary)
display(summary_df)



Processing Radiom | ICU
Before: {0: 1187, 1: 493}
After: {1: 1156, 0: 14}

Processing Radiom | mechanical_ventilation
Before: {0: 1224, 1: 456}
After: {1: 1190, 0: 28}

Processing Radiom | death
Before: {0: 1409, 1: 271}
After: {1: 1407, 0: 35}

Processing RadioClin | ICU
Before: {0: 1187, 1: 493}
After: {1: 1163, 0: 13}

Processing RadioClin | mechanical_ventilation
Before: {0: 1224, 1: 456}
After: {1: 1203, 0: 26}

Processing RadioClin | death
Before: {0: 1409, 1: 271}
After: {1: 1407, 0: 27}

Processing RadioClinLab | ICU
Before: {0: 1187, 1: 493}
After: {1: 1153, 0: 19}

Processing RadioClinLab | mechanical_ventilation
Before: {0: 1224, 1: 456}
After: {1: 1207, 0: 34}

Processing RadioClinLab | death
Before: {0: 1409, 1: 271}
After: {1: 1409, 0: 32}


,Feature Group,Target,Original Rows,Resampled Rows,Original Positives,Resampled Positives
0,Radiom,ICU,1680,1170,493,1156
1,Radiom,mechanical_ventilation,1680,1218,456,1190
2,Radiom,death,1680,1442,271,1407
3,RadioClin,ICU,1680,1176,493,1163
4,RadioClin,mechanical_ventilation,1680,1229,456,1203
5,RadioClin,death,1680,1434,271,1407
6,RadioClinLab,ICU,1680,1172,493,1153
7,RadioClinLab,mechanical_ventilation,1680,1241,456,1207
8,RadioClinLab,death,1680,1441,271,1409


In [16]:

from pathlib import Path
from scipy import sparse
import numpy as np

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/Colab Notebooks/data/resampled"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def save_matrix(matrix, path):
    if sparse.issparse(matrix):
        sparse.save_npz(str(path) + ".npz", matrix)
    else:
        np.save(str(path) + ".npy", matrix)

for (group, target), (X_resampled, y_resampled) in resampled_data.items():
    name = f"{group}_{target}"

    save_matrix(
        X_resampled,
        OUTPUT_DIR / f"{name}_X_train_resampled"
    )

    np.save(
        OUTPUT_DIR / f"{name}_y_train_resampled.npy",
        np.asarray(y_resampled)
    )

summary_df.to_csv(
    OUTPUT_DIR / "smoteenn_summary.csv",
    index=False
)

print("Resampled datasets and summary saved to:", OUTPUT_DIR)


Resampled datasets and summary saved to: /content/drive/MyDrive/Colab Notebooks/data/resampled
